In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps as UTC
metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

log_df = pd.read_csv("log.csv")
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

error_df = pd.read_csv("error_logs.csv")
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Helper to build the compact aggregate table for metric-like files
def build_aggregate_table(df, key_cols, value_col, name_cols_limit=20):
    g = df.groupby(key_cols)[value_col]
    agg = g.agg(
        count_of_points='count',
        global_P50=lambda x: float(x.quantile(0.50)) if x.size>0 else float('nan'),
        global_P90=lambda x: float(x.quantile(0.90)) if x.size>0 else float('nan'),
        global_P95=lambda x: float(x.quantile(0.95)) if x.size>0 else float('nan'),
        min_value='min',
        max_value='max'
    ).reset_index()
    # sort for determinism and compactness: by count desc, then max_value desc
    agg = agg.sort_values(['count_of_points', 'max_value'], ascending=[False, False]).reset_index(drop=True)
    return agg.head(name_cols_limit), agg

# Metric aggregates
metric_summary_head, metric_agg_full = build_aggregate_table(metric_df, ['cmdb_id', 'kpi_name'], 'value', 20)
metric_earliest = metric_df['timestamp'].min()
metric_latest = metric_df['timestamp'].max()
metric_unique_cmdb = metric_df['cmdb_id'].nunique()
metric_unique_kpi = metric_df['kpi_name'].nunique()

# Trace aggregates
trace_summary_head, trace_agg_full = build_aggregate_table(trace_df, ['cmdb_id', 'trace_name'], 'value', 20)
trace_earliest = trace_df['timestamp'].min()
trace_latest = trace_df['timestamp'].max()
trace_unique_cmdb = trace_df['cmdb_id'].nunique()
trace_unique_name = trace_df['trace_name'].nunique()

# Log aggregates
log_summary_head, log_agg_full = build_aggregate_table(log_df, ['cmdb_id', 'log_name'], 'value', 20)
log_earliest = log_df['timestamp'].min()
log_latest = log_df['timestamp'].max()
log_unique_cmdb = log_df['cmdb_id'].nunique()
log_unique_name = log_df['log_name'].nunique()

# Error logs summary and sample (up to 5 rows sorted by timestamp ascending)
error_total_rows = len(error_df)
error_earliest = error_df['timestamp'].min() if error_total_rows>0 else pd.NaT
error_latest = error_df['timestamp'].max() if error_total_rows>0 else pd.NaT
error_unique_cmdb = error_df['cmdb_id'].nunique()
error_samples = error_df.sort_values('timestamp')[['timestamp','cmdb_id','message']].head(5)

# Compact displays: selected aggregates + summary values
metric_summary_head, metric_earliest, metric_latest, metric_unique_cmdb, metric_unique_kpi, \
trace_summary_head, trace_earliest, trace_latest, trace_unique_cmdb, trace_unique_name, \
log_summary_head, log_earliest, log_latest, log_unique_cmdb, log_unique_name, \
error_total_rows, error_earliest, error_latest, error_unique_cmdb, error_samples

```
Out[1]:
```
summary = (
    "Summary of telemetry files (based on the loaded CSVs):\n\n"
    "metric.csv:\n"
    "- Time range: 2024-01-25 03:43:00 UTC to 2024-01-25 04:07:00 UTC.\n"
    "- Unique cmdb_id: 68. Unique kpi_name: 8.\n"
    "- Data is dominated by memory KPIs (kpi_name 'mem') with 25 points per (cmdb_id,kpi_name) in the shown top 20.\n"
    "- Memory values in the top rows are large (~2.5e8 up to ~6.0e8); each listed pair has count_of_points=25.\n\n"
    "trace.csv:\n"
    "- Time range: 2024-01-25 03:43:00 UTC to 2024-01-25 04:07:00 UTC.\n"
    "- Unique cmdb_id: 28. Unique trace_name: 220.\n"
    "- Top trace entries (shown) are per-service row_count traces (e.g., ts-station-service) with medians ranging from ~1e3 to ~2.7e4 and count_of_points=25.\n\n"
    "log.csv:\n"
    "- Time range: 2024-01-25 03:43:00 UTC to 2024-01-25 04:07:00 UTC.\n"
    "- Unique cmdb_id: 46. Unique log_name: 2.\n"
    "- Shown top log metric is 'log.row_count' with median values up to ~2563 (ts-basic-service) and count_of_points=25 for listed rows.\n\n"
    "error_logs.csv:\n"
    "- Total rows: 0. No timestamps or samples available.\n\n"
    "Overall: metric, trace, and log files cover the same 24-minute window with complete per-minute series (25 points each for the shown pairs). There are no error log entries."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id kpi_name  count_of_points    global_P50    global_P90    global_P95     min_value     max_value
0            ts-voucher-mysql      mem               25  6.004204e+08  6.004204e+08  6.004204e+08  6.004204e+08  6.004204e+08
1               ts-food-mysql      mem               25  5.998100e+08  5.998100e+08  5.998100e+08  5.998100e+08  5.998100e+08
2           ts-travel-service      mem               25  2.937405e+08  2.938659e+08  2.938958e+08  2.934342e+08  2.939031e+08
3          ts-travel2-service      mem               25  2.818384e+08  2.819880e+08  2.820085e+08  2.816325e+08  2.820211e+08
4            ts-order-service      mem               25  2.745019e+08  2.802966e+08  2.809354e+08  2.694236e+08  2.815767e+08
5      ts-order-other-service      mem               25  2.709393e+08  2.723124e+08  2.723843e+08  2.693480e+08  2.725478e+08
6          ts-station-service      mem               25  2.654837e+08  2.682084e+08  2.688100e+08  2.654196e+08  2.690171e+08
7             ts-food-service      mem               25  2.611963e+08  2.614894e+08  2.615151e+08  2.606776e+08  2.615378e+08
8         ts-food-map-service      mem               25  2.587482e+08  2.594852e+08  2.596029e+08  2.581800e+08  2.597177e+08
9            ts-route-service      mem               25  2.571176e+08  2.571971e+08  2.572080e+08  2.570269e+08  2.572331e+08
10          ts-config-service      mem               25  2.567851e+08  2.568495e+08  2.568598e+08  2.561863e+08  2.569033e+08
11           ts-train-service      mem               25  2.553230e+08  2.563050e+08  2.563120e+08  2.524257e+08  2.563508e+08
12           ts-price-service      mem               25  2.549395e+08  2.550302e+08  2.550507e+08  2.548914e+08  2.550622e+08
13  ts-inside-payment-service      mem               25  2.535501e+08  2.537915e+08  2.538038e+08  2.532051e+08  2.538252e+08
14      ts-ticketinfo-service      mem               25  2.529724e+08  2.530119e+08  2.530208e+08  2.529036e+08  2.530674e+08
15        ts-contacts-service      mem               25  2.516383e+08  2.524320e+08  2.524744e+08  2.509393e+08  2.524843e+08
16       ts-assurance-service      mem               25  2.505985e+08  2.514481e+08  2.517169e+08  2.493952e+08  2.517606e+08
17        ts-security-service      mem               25  2.510986e+08  2.514159e+08  2.514230e+08  2.505716e+08  2.514323e+08
18         ts-consign-service      mem               25  2.497122e+08  2.499104e+08  2.499586e+08  2.493723e+08  2.499789e+08
19           ts-basic-service      mem               25  2.498531e+08  2.498987e+08  2.499020e+08  2.498123e+08  2.499034e+08, Timestamp('2024-01-25 03:43:00+0000', tz='UTC'), Timestamp('2024-01-25 04:07:00+0000', tz='UTC'), 68, 8,                   cmdb_id                                  trace_name  count_of_points  global_P50  global_P90  global_P95  min_value  max_value
0      ts-station-service     trace.from_ts-station-service.row_count               25     27258.0     35416.4     36714.0    10814.0    40060.0
1      ts-station-service       trace.to_ts-station-service.row_count               25     27258.0     35416.4     36714.0    10814.0    40060.0
2      ts-travel2-service     trace.from_ts-travel2-service.row_count               25      2658.0      2904.4      2918.8      837.0     2933.0
3      ts-travel2-service       trace.to_ts-travel2-service.row_count               25      2658.0      2904.4      2918.8      837.0     2933.0
4        ts-train-service       trace.from_ts-train-service.row_count               25      2592.0      2800.0      2833.6      836.0     2896.0
5        ts-train-service         trace.to_ts-train-service.row_count               25      2592.0      2800.0      2833.6      836.0     2896.0
6        ts-route-service       trace.from_ts-route-service.row_count               25      2096.0      2250.0      2284.4      676.0     2314.0
7        ts-route-service         trace.to_ts-route-service.row_count               25      2096.0      2250.0      2284.4      676.0     2314.0
8        ts-basic-service       trace.from_ts-basic-service.row_count               25      1853.0      2035.0      2040.6      566.0     2115.0
9        ts-basic-service         trace.to_ts-basic-service.row_count               25      1853.0      2035.0      2040.6      566.0     2115.0
10      ts-travel-service      trace.from_ts-travel-service.row_count               25      1566.0      1843.2      1849.6      502.0     2068.0
11      ts-travel-service        trace.to_ts-travel-service.row_count               25      1566.0      1843.2      1849.6      502.0     2068.0
12       ts-basic-service       trace.to_ts-station-service.row_count               25      1435.0      1569.6      1573.2      440.0     1626.0
13     ts-station-service       trace.from_ts-basic-service.row_count               25      1435.0      1569.6      1573.2      440.0     1626.0
14        ts-seat-service        trace.from_ts-seat-service.row_count               25      1096.0      1193.4      1200.2      341.0     1234.0
15        ts-seat-service          trace.to_ts-seat-service.row_count               25      1096.0      1193.4      1200.2      341.0     1234.0
16  ts-ticketinfo-service  trace.from_ts-ticketinfo-service.row_count               25      1017.0      1105.2      1106.8      314.0     1138.0
17  ts-ticketinfo-service    trace.to_ts-ticketinfo-service.row_count               25      1017.0      1105.2      1106.8      314.0     1138.0
18       ts-basic-service  trace.from_ts-ticketinfo-service.row_count               25      1017.0      1105.6      1106.8      314.0     1137.0
19  ts-ticketinfo-service         trace.to_ts-basic-service.row_count               25      1017.0      1105.6      1106.8      314.0     1137.0, Timestamp('2024-01-25 03:43:00+0000', tz='UTC'), Timestamp('2024-01-25 04:07:00+0000', tz='UTC'), 28, 220,                         cmdb_id       log_name  count_of_points  global_P50  global_P90  global_P95  min_value  max_value
0              ts-basic-service  log.row_count               25      2563.0      2819.2      2823.6        776       2930
1            ts-travel2-service  log.row_count               25      2346.0      2534.4      2548.8        740       2583
2             ts-travel-service  log.row_count               25      1753.0      2071.2      2074.4        574       2325
3               ts-seat-service  log.row_count               25      1907.0      2094.8      2113.6        588       2164
4              ts-route-service  log.row_count               25      1048.0      1125.0      1142.2        338       1157
5        ts-order-other-service  log.row_count               25       463.0       529.0       536.6        139        539
6              ts-order-service  log.row_count               25       391.0       459.6       465.2        116        467
7               ts-auth-service  log.row_count               25       336.0       355.2       359.2        116        370
8               ts-ui-dashboard  log.row_count               25       288.0       305.2       322.0         93        329
9              ts-price-service  log.row_count               25       284.0       312.0       315.2         84        326
10          ts-preserve-service  log.row_count               25       111.0       194.2       222.2         27        275
11              ts-food-service  log.row_count               25        77.0        91.6       103.2         18        112
12          ts-security-service  log.row_count               25        56.0       100.8       110.4          8        112
13           ts-consign-service  log.row_count               25        47.0        74.2        82.6         14         91
14          ts-contacts-service  log.row_count               25        42.0        75.6        82.8          6         84
15          ts-food-map-service  log.row_count               25        23.0        27.6        31.2          6         34
16  ts-admin-basic-info-service  log.row_count               25        28.0        32.6        33.0         10         33
17      ts-admin-travel-service  log.row_count               25        27.0        30.0        31.6         12         33
18    ts-inside-payment-service  log.row_count               25        14.0        20.0        21.6          6         26
19         ts-assurance-service  log.row_count               25        10.0        19.2        20.0          2         21, Timestamp('2024-01-25 03:43:00+0000', tz='UTC'), Timestamp('2024-01-25 04:07:00+0000', tz='UTC'), 46, 2, 0, NaT, NaT, 0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing dataframes if present; otherwise load and parse timestamps
try:
    metric_df
    trace_df
    log_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    trace_df = pd.read_csv("trace.csv")
    log_df = pd.read_csv("log.csv")
    error_df = pd.read_csv("error_logs.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
else:
    # ensure timestamps are parsed per instructions
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Candidate components
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Helper to compute global aggregates for a df grouped by given name_col (kpi/trace/log)
def compute_global_aggs(df, group_keys, value_col):
    g = df.groupby(group_keys)[value_col]
    agg = g.agg(
        count_of_points='count',
        global_P05=lambda x: float(x.quantile(0.05)),
        global_P50=lambda x: float(x.quantile(0.50)),
        global_P90=lambda x: float(x.quantile(0.90)),
        global_P95=lambda x: float(x.quantile(0.95)),
        min_value='min',
        max_value='max'
    ).reset_index()
    return agg

# 1) Metric global aggregates for candidate components
metric_candidates_df = metric_df[metric_df['cmdb_id'].isin(candidates)]
metric_agg = compute_global_aggs(metric_candidates_df, ['cmdb_id','kpi_name'], 'value')

# 2) Trace global aggregates for candidate components
trace_candidates_df = trace_df[trace_df['cmdb_id'].isin(candidates)]
trace_agg = compute_global_aggs(trace_candidates_df, ['cmdb_id','trace_name'], 'value')

# 3) Log global aggregates for candidate components
log_candidates_df = log_df[log_df['cmdb_id'].isin(candidates)]
log_agg = compute_global_aggs(log_candidates_df, ['cmdb_id','log_name'], 'value')

# Incident window (UTC)
start = pd.to_datetime('2024-01-25 03:40:18+00:00')
end   = pd.to_datetime('2024-01-25 04:10:18+00:00')

# Filter original dataframes to incident window and candidates
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end) & (metric_df['cmdb_id'].isin(candidates))]
trace_window  = trace_df[(trace_df['timestamp']  >= start) & (trace_df['timestamp']  <= end) & (trace_df['cmdb_id'].isin(candidates))]
log_window    = log_df[(log_df['timestamp']    >= start) & (log_df['timestamp']    <= end) & (log_df['cmdb_id'].isin(candidates))]

# Function to detect anomalies and build compact report for a source
def build_report(global_agg, window_df, group_name_col, value_col, source_label):
    # Merge window rows with global thresholds
    key_cols = ['cmdb_id', group_name_col]
    joined = window_df.merge(global_agg, on=key_cols, how='left', suffixes=('','_glob'))
    if joined.empty:
        # No window data for any group
        # Build report rows from global_agg only, with zero anomalies and NaT for timestamps and NaN for window min/max
        report = global_agg.rename(columns={group_name_col: 'name'}).copy()
        report = report.rename(columns={'count_of_points':'count_of_points_full',
                                        'global_P05':'global_P05','global_P50':'global_P50',
                                        'global_P90':'global_P90','global_P95':'global_P95',
                                        'min_value':'min_value_full','max_value':'max_value_full'})
        report['count_anomalies_in_window'] = 0
        report['earliest_anomaly_timestamp_in_window'] = pd.NaT
        report['max_value_in_window'] = pd.NA
        report['min_value_in_window'] = pd.NA
        report['source'] = source_label
        return report
    # anomaly condition per-row
    joined['is_anomaly'] = (joined[value_col] >= joined['global_P95']) | (joined[value_col] <= joined['global_P05'])
    # group anomalies
    anomalies_grouped = joined[joined['is_anomaly']].groupby(key_cols).agg(
        count_anomalies_in_window=(value_col,'count'),
        earliest_anomaly_timestamp_in_window=('timestamp','min')
    ).reset_index()
    # window stats per group
    window_stats = joined.groupby(key_cols).agg(
        max_value_in_window=(value_col,'max'),
        min_value_in_window=(value_col,'min'),
        count_points_in_window=(value_col,'count')
    ).reset_index()
    # build base report from global agg
    report = global_agg.copy()
    # rename group name column to a generic name for display
    report = report.rename(columns={group_name_col:'name',
                                    'count_of_points':'count_of_points_full',
                                    'min_value':'min_value_full',
                                    'max_value':'max_value_full',
                                    'global_P05':'global_P05',
                                    'global_P50':'global_P50',
                                    'global_P90':'global_P90',
                                    'global_P95':'global_P95'})
    # merge window-derived stats and anomalies
    report = report.merge(window_stats.rename(columns={group_name_col:'name'}), on=['cmdb_id','name'], how='left')
    report = report.merge(anomalies_grouped.rename(columns={group_name_col:'name'}), on=['cmdb_id','name'], how='left')
    # fill NaNs for groups without anomalies/window data
    report['count_anomalies_in_window'] = report['count_anomalies_in_window'].fillna(0).astype(int)
    report['earliest_anomaly_timestamp_in_window'] = pd.to_datetime(report['earliest_anomaly_timestamp_in_window'])
    # ensure window min/max are present (NaN if no rows)
    # add source label
    report['source'] = source_label
    # reorder/select columns per requirement
    report = report[['cmdb_id','name','count_of_points_full','global_P05','global_P50','global_P90','global_P95',
                     'count_anomalies_in_window','earliest_anomaly_timestamp_in_window','max_value_in_window','min_value_in_window','source']]
    return report

# Build reports
metric_report = build_report(metric_agg, metric_window, 'kpi_name', 'value', 'metric')
trace_report  = build_report(trace_agg, trace_window, 'trace_name', 'value', 'trace')
log_report    = build_report(log_agg, log_window, 'log_name', 'value', 'log')

# Limit reports to 20 rows for display
metric_report_head = metric_report.head(20)
trace_report_head  = trace_report.head(20)
log_report_head    = log_report.head(20)

# Build combined anomalies table across all three sources (rows within window that are anomalies)
def extract_anomalies(window_df, global_agg, group_name_col, value_col, source_label):
    if window_df.empty:
        return pd.DataFrame(columns=['source','cmdb_id','name','timestamp','value'])
    joined = window_df.merge(global_agg, on=['cmdb_id', group_name_col], how='left')
    joined['is_anomaly'] = (joined[value_col] >= joined['global_P95']) | (joined[value_col] <= joined['global_P05'])
    anom = joined[joined['is_anomaly']].copy()
    if anom.empty:
        return pd.DataFrame(columns=['source','cmdb_id','name','timestamp','value'])
    anom = anom.rename(columns={group_name_col:'name', value_col:'value'})[['cmdb_id','name','timestamp','value']]
    anom['source'] = source_label
    # reorder
    anom = anom[['source','cmdb_id','name','timestamp','value']]
    return anom

anom_metric_rows = extract_anomalies(metric_window, metric_agg, 'kpi_name', 'value', 'metric')
anom_trace_rows  = extract_anomalies(trace_window, trace_agg, 'trace_name', 'value', 'trace')
anom_log_rows    = extract_anomalies(log_window, log_agg, 'log_name', 'value', 'log')

combined_anomalies = pd.concat([anom_metric_rows, anom_trace_rows, anom_log_rows], ignore_index=True)
# sort by timestamp ascending and limit to top 20
if not combined_anomalies.empty:
    combined_anomalies = combined_anomalies.sort_values('timestamp').head(20)
else:
    combined_anomalies = combined_anomalies

# Note if any candidate components have zero data in a source
missing_in_metric = [c for c in candidates if c not in metric_agg['cmdb_id'].unique()]
missing_in_trace  = [c for c in candidates if c not in trace_agg['cmdb_id'].unique()]
missing_in_log    = [c for c in candidates if c not in log_agg['cmdb_id'].unique()]

missing_notes = {
    'metric_missing_components': missing_in_metric,
    'trace_missing_components': missing_in_trace,
    'log_missing_components': missing_in_log
}

# Compact outputs (limited). Return the three reports (head), combined anomalies top20, and missing notes.
metric_report_head, trace_report_head, log_report_head, combined_anomalies, missing_notes

```
Out[2]:
```
Summary of findings (plain English):

- Data coverage:
  - All five candidate components (ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service) have data in metric, trace, and log sources (no component is missing from any source).

- Time window and anomaly detection rule:
  - Incident window used: 2024-01-25 03:40:18 → 2024-01-25 04:10:18 UTC.
  - A point is flagged as anomalous if its value >= global P95 or <= global P05 (global percentiles computed from the full series per component/KPI).

- Key summary per component (high-level):
  - ts-auth-service:
    - Metrics: multiple KPIs (cpu, diskio, latency-50, latency-90, mem, socket, workload) show anomalies in the incident window (4–5 anomalous points each; earliest anomalies near 03:43–03:47 UTC).
    - Traces: duration_mean, duration_p95, row_count and related trace metrics show anomalies (earliest ~03:44–03:45 UTC).
    - Logs: log.error_count is uniformly zero (flagged as anomalous across full series), log.row_count shows anomalies (earliest ~03:43 UTC).
    - Notable: memory, disk I/O and latency spikes present; earliest anomalous timestamps around 03:43–03:47 UTC.

  - ts-order-service:
    - Metrics: cpu, diskio, latency-50/90, mem, socket, workload show anomalies (typically 4 anomalous points each). The "error" metric is zero across series and thus counted as anomalous for all points (25).
    - Traces: many trace metrics (from_root.* and service-to-service traces) show anomalies (earliest anomalies ~03:43–03:47 UTC).
    - Logs: log.error_count all-zero (25 anomalies); log.row_count shows anomalies (earliest ~03:44 UTC).
    - Notable: disk I/O and latency spikes and repeated zero error metrics.

  - ts-route-service:
    - Metrics: cpu, diskio, latency-50/90, mem, socket/workload show anomalies (multiple points). The "error" metric is zero across series (25 anomalies).
    - Traces: several trace metrics show anomalies; earliest anomaly times around 03:43–03:45 UTC.
    - Logs: log.row_count and log.error_count show anomalies (error_count all-zero case flagged).
    - Notable: similar pattern of latency/disk/memory anomalies plus zero error counters.

  - ts-train-service:
    - Metrics/traces/logs: present and show anomalies (trace row_count values and many trace error_rate rows at 03:43 are flagged). Logs have single-point entries (one row point at 03:54 UTC) and are present.
    - Notable: some large trace row_count values at the start of the window (03:43 UTC) flagged.

  - ts-travel-service:
    - Metrics/traces/logs: present. Many trace rows (including various from_/to_ traces) at the window start (03:43 UTC) are flagged; log.row_count shows anomalies (earliest ~03:50 UTC).
    - Notable: multiple trace anomaly entries at 03:43 UTC.

- Global patterns and most salient anomalies:
  - Many anomalies appear at or near 2024-01-25 03:43:00 UTC (start of the window) across traces and metrics for multiple services — many trace.error_rate and log.error_count values are exactly 0 and were flagged because P05 or P95 thresholds equal those values (these "all-zero" series produce anomaly counts of 25).
  - ts-auth-service, ts-order-service and ts-route-service show the most and the broadest anomalies across metric, trace, and log sources (multiple KPIs: memory, disk I/O, latency, CPU, etc.).
  - The combined anomaly list includes a metric mem spike for ts-auth-service at 03:43 and numerous trace anomalies for train/travel services at 03:43.

- Earliest anomalous timestamps:
  - Earliest anomalies generally start at 2024-01-25 03:43:00 UTC (multiple services / metrics / traces); other anomalies appear through ~03:50 UTC.

- Recommendation (operational summary):
  - Investigate ts-auth-service, ts-order-service, and ts-route-service first (they show concurrent anomalies across metrics, traces, and logs — memory/disk/latency/CPU signals).
  - Pay attention to:
    - Memory and disk I/O spikes,
    - Latency (p50/p90/p95) increases,
    - The all-zero error counters in error metrics/logs/trace error_rate (these may reflect measurement issues or genuine zero errors; treat the all-zero condition carefully).
  - Investigate the cluster/system behavior around ~03:43 UTC when many anomalies first appear.

If you want, I can produce a concise table of the top anomalous component-KPI rows (timestamps and values) or expand the per-component anomaly counts and earliest times.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id        name  count_of_points_full    global_P05    global_P50    global_P90    global_P95  count_anomalies_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  min_value_in_window  source
0    ts-auth-service         cpu                    25  1.315960e+01  1.487895e+01  1.583560e+01  1.644762e+01                          4            2024-01-25 03:47:00+00:00         1.707926e+01         1.243436e+01  metric
1    ts-auth-service      diskio                    25  4.200346e+04  5.191844e+04  6.030125e+04  6.058071e+04                          4            2024-01-25 03:49:00+00:00         6.495578e+04         3.618975e+04  metric
2    ts-auth-service  latency-50                    25  2.100021e-01  2.383054e-01  2.697558e-01  2.702664e-01                          4            2024-01-25 03:53:00+00:00         2.862363e-01         2.056077e-01  metric
3    ts-auth-service  latency-90                    25  4.787485e-01  6.754030e-01  8.424909e-01  8.759720e-01                          4            2024-01-25 03:44:00+00:00         8.887364e-01         4.541015e-01  metric
4    ts-auth-service         mem                    25  2.479559e+08  2.483359e+08  2.489667e+08  2.490065e+08                          4            2024-01-25 03:43:00+00:00         2.491505e+08         2.479418e+08  metric
5    ts-auth-service      socket                    25  1.380000e+01  1.468333e+01  1.562667e+01  1.575333e+01                          5            2024-01-25 03:46:00+00:00         1.600000e+01         1.323333e+01  metric
6    ts-auth-service    workload                    25  2.568050e+00  2.841357e+00  2.947043e+00  2.981463e+00                          4            2024-01-25 03:47:00+00:00         3.000000e+00         2.461167e+00  metric
7   ts-order-service         cpu                    25  1.754465e+00  2.465441e+00  2.947879e+00  3.025231e+00                          4            2024-01-25 03:49:00+00:00         3.196939e+00         1.541157e+00  metric
8   ts-order-service      diskio                    25  3.996594e+04  5.336315e+04  5.795890e+04  6.232581e+04                          4            2024-01-25 03:50:00+00:00         6.746179e+04         3.610141e+04  metric
9   ts-order-service       error                    25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00                         25            2024-01-25 03:43:00+00:00         0.000000e+00         0.000000e+00  metric
10  ts-order-service  latency-50                    25  7.318014e-03  8.223337e-03  9.725317e-03  9.991070e-03                          4            2024-01-25 03:50:00+00:00         1.140464e-02         7.210625e-03  metric
11  ts-order-service  latency-90                    25  2.195181e-02  2.331956e-02  2.495336e-02  2.616295e-02                          4            2024-01-25 03:44:00+00:00         3.625866e-01         2.107269e-02  metric
12  ts-order-service         mem                    25  2.695580e+08  2.745019e+08  2.802966e+08  2.809354e+08                          4            2024-01-25 03:43:00+00:00         2.815767e+08         2.694236e+08  metric
13  ts-order-service      socket                    25  1.040333e+01  1.154762e+01  1.264667e+01  1.270667e+01                          4            2024-01-25 03:46:00+00:00         1.316667e+01         1.023333e+01  metric
14  ts-order-service    workload                    25  5.191680e+00  6.532283e+00  7.609487e+00  7.734433e+00                          4            2024-01-25 03:50:00+00:00         7.951133e+00         4.559900e+00  metric
15  ts-route-service         cpu                    25  4.575962e+00  5.021562e+00  5.411188e+00  5.487051e+00                          4            2024-01-25 03:45:00+00:00         5.571167e+00         3.481604e+00  metric
16  ts-route-service      diskio                    25  4.001961e+04  5.205796e+04  5.865046e+04  6.047917e+04                          4            2024-01-25 03:49:00+00:00         6.096313e+04         2.021842e+04  metric
17  ts-route-service       error                    25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00                         25            2024-01-25 03:43:00+00:00         0.000000e+00         0.000000e+00  metric
18  ts-route-service  latency-50                    25  3.785430e-03  4.092436e-03  4.312852e-03  4.338656e-03                          4            2024-01-25 03:45:00+00:00         4.464658e-03         3.736852e-03  metric
19  ts-route-service  latency-90                    25  8.430856e-03  8.847707e-03  9.050294e-03  9.070183e-03                          4            2024-01-25 03:52:00+00:00         9.074807e-03         8.368080e-03  metric,              cmdb_id                                               name  count_of_points_full  global_P05  global_P50  global_P90  global_P95  count_anomalies_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  min_value_in_window source
0    ts-auth-service                      trace.from_root.duration_mean                    25    0.254567    0.326183    0.369500    0.375588                          4            2024-01-25 03:44:00+00:00             0.388194             0.252508  trace
1    ts-auth-service                       trace.from_root.duration_p95                    25    0.483937    0.666962    1.059668    1.122342                          4            2024-01-25 03:45:00+00:00             1.241927             0.471851  trace
2    ts-auth-service                         trace.from_root.error_rate                    25    0.000000    0.000000    0.000000    0.000000                         25            2024-01-25 03:43:00+00:00             0.000000             0.000000  trace
3    ts-auth-service                          trace.from_root.row_count                    25   61.600000   84.000000   89.600000   90.000000                          5            2024-01-25 03:43:00+00:00            92.000000            29.000000  trace
4    ts-auth-service           trace.from_ts-auth-service.duration_mean                    25    0.001775    0.002571    0.003223    0.003393                          4            2024-01-25 03:54:00+00:00             0.004252             0.001605  trace
5    ts-auth-service            trace.from_ts-auth-service.duration_p95                    25    0.001807    0.002221    0.002750    0.003038                          4            2024-01-25 03:44:00+00:00             0.005109             0.001786  trace
6    ts-auth-service              trace.from_ts-auth-service.error_rate                    25    0.000000    0.000000    0.000000    0.000000                         25            2024-01-25 03:43:00+00:00             0.000000             0.000000  trace
7    ts-auth-service               trace.from_ts-auth-service.row_count                    25  243.600000  336.000000  355.200000  359.200000                          4            2024-01-25 03:43:00+00:00           370.000000           116.000000  trace
8    ts-auth-service             trace.to_ts-auth-service.duration_mean                    25    0.001775    0.002571    0.003223    0.003393                          4            2024-01-25 03:54:00+00:00             0.004252             0.001605  trace
9    ts-auth-service              trace.to_ts-auth-service.duration_p95                    25    0.001807    0.002221    0.002750    0.003038                          4            2024-01-25 03:44:00+00:00             0.005109             0.001786  trace
10   ts-auth-service                trace.to_ts-auth-service.error_rate                    25    0.000000    0.000000    0.000000    0.000000                         25            2024-01-25 03:43:00+00:00             0.000000             0.000000  trace
11   ts-auth-service                 trace.to_ts-auth-service.row_count                    25  243.600000  336.000000  355.200000  359.200000                          4            2024-01-25 03:43:00+00:00           370.000000           116.000000  trace
12  ts-order-service                      trace.from_root.duration_mean                    25    0.773318    0.855844    0.945140    0.953586                          4            2024-01-25 03:43:00+00:00             0.969506             0.757687  trace
13  ts-order-service                       trace.from_root.duration_p95                    25    0.842345    0.923266    1.019010    1.042775                          4            2024-01-25 03:46:00+00:00             1.081379             0.831440  trace
14  ts-order-service                         trace.from_root.error_rate                    25    0.000000    0.000000    0.000000    0.000000                         25            2024-01-25 03:43:00+00:00             0.000000             0.000000  trace
15  ts-order-service                          trace.from_root.row_count                    25    4.000000    8.000000   12.000000   14.400000                          6            2024-01-25 03:47:00+00:00            18.000000             3.000000  trace
16  ts-order-service  trace.from_ts-inside-payment-service.duration_...                    25    0.003279    0.003942    0.004574    0.004917                          4            2024-01-25 03:47:00+00:00             0.004988             0.003037  trace
17  ts-order-service  trace.from_ts-inside-payment-service.duration_p95                    25    0.003567    0.004760    0.007116    0.008552                          4            2024-01-25 03:47:00+00:00             0.008975             0.003060  trace
18  ts-order-service    trace.from_ts-inside-payment-service.error_rate                    25    0.000000    0.000000    0.000000    0.000000                         25            2024-01-25 03:43:00+00:00             0.000000             0.000000  trace
19  ts-order-service     trace.from_ts-inside-payment-service.row_count                    25    2.000000    6.000000   10.000000   11.600000                          5            2024-01-25 03:47:00+00:00            13.000000             2.000000  trace,              cmdb_id             name  count_of_points_full  global_P05  global_P50  global_P90  global_P95  count_anomalies_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  min_value_in_window source
0    ts-auth-service  log.error_count                    25         0.0         0.0         0.0         0.0                         25            2024-01-25 03:43:00+00:00                    0                    0    log
1    ts-auth-service    log.row_count                    25       244.4       336.0       355.2       359.2                          4            2024-01-25 03:43:00+00:00                  370                  116    log
2   ts-order-service  log.error_count                    25         0.0         0.0         0.0         0.0                         25            2024-01-25 03:43:00+00:00                    0                    0    log
3   ts-order-service    log.row_count                    25       302.0       391.0       459.6       465.2                          4            2024-01-25 03:44:00+00:00                  467                  116    log
4   ts-route-service  log.error_count                    25         0.0         0.0         0.0         0.0                         25            2024-01-25 03:43:00+00:00                    0                    0    log
5   ts-route-service    log.row_count                    25       779.4      1048.0      1125.0      1142.2                          4            2024-01-25 03:43:00+00:00                 1157                  338    log
6   ts-train-service  log.error_count                     1         0.0         0.0         0.0         0.0                          1            2024-01-25 03:54:00+00:00                    0                    0    log
7   ts-train-service    log.row_count                     1       162.0       162.0       162.0       162.0                          1            2024-01-25 03:54:00+00:00                  162                  162    log
8  ts-travel-service  log.error_count                    25         0.0         0.0         0.0         0.0                         25            2024-01-25 03:43:00+00:00                    0                    0    log
9  ts-travel-service    log.row_count                    25      1311.2      1753.0      2071.2      2074.4                          4            2024-01-25 03:50:00+00:00                 2325                  574    log,      source            cmdb_id                                           name                 timestamp         value
0    metric    ts-auth-service                                            mem 2024-01-25 03:43:00+00:00  2.479445e+08
264   trace   ts-train-service         trace.from_ts-train-service.error_rate 2024-01-25 03:43:00+00:00  0.000000e+00
265   trace   ts-train-service          trace.from_ts-train-service.row_count 2024-01-25 03:43:00+00:00  1.828000e+03
266   trace   ts-train-service        trace.from_ts-travel-service.error_rate 2024-01-25 03:43:00+00:00  0.000000e+00
267   trace   ts-train-service       trace.from_ts-travel2-service.error_rate 2024-01-25 03:43:00+00:00  0.000000e+00
268   trace   ts-train-service        trace.from_ts-travel2-service.row_count 2024-01-25 03:43:00+00:00  1.900000e+02
269   trace   ts-train-service           trace.to_ts-train-service.error_rate 2024-01-25 03:43:00+00:00  0.000000e+00
270   trace   ts-train-service            trace.to_ts-train-service.row_count 2024-01-25 03:43:00+00:00  1.828000e+03
271   trace  ts-travel-service                     trace.from_root.error_rate 2024-01-25 03:43:00+00:00  0.000000e+00
272   trace  ts-travel-service  trace.from_ts-admin-travel-service.error_rate 2024-01-25 03:43:00+00:00  0.000000e+00
273   trace  ts-travel-service   trace.from_ts-admin-travel-service.row_count 2024-01-25 03:43:00+00:00  8.000000e+00
274   trace  ts-travel-service          trace.from_ts-food-service.error_rate 2024-01-25 03:43:00+00:00  0.000000e+00
275   trace  ts-travel-service   trace.from_ts-preserve-service.duration_mean 2024-01-25 03:43:00+00:00  2.870428e-01
276   trace  ts-travel-service    trace.from_ts-preserve-service.duration_p95 2024-01-25 03:43:00+00:00  2.941040e-01
263   trace   ts-train-service          trace.from_ts-basic-service.row_count 2024-01-25 03:43:00+00:00  9.700000e+01
277   trace  ts-travel-service      trace.from_ts-preserve-service.error_rate 2024-01-25 03:43:00+00:00  0.000000e+00
279   trace  ts-travel-service          trace.from_ts-seat-service.error_rate 2024-01-25 03:43:00+00:00  0.000000e+00
280   trace  ts-travel-service        trace.from_ts-travel-service.error_rate 2024-01-25 03:43:00+00:00  0.000000e+00
281   trace  ts-travel-service           trace.to_ts-order-service.error_rate 2024-01-25 03:43:00+00:00  0.000000e+00
282   trace  ts-travel-service           trace.to_ts-route-service.error_rate 2024-01-25 03:43:00+00:00  0.000000e+00, {'metric_missing_components': [], 'trace_missing_components': [], 'log_missing_components': []})```
```

In [ ]:
# In[3]:

import pandas as pd
import re
# Reuse existing trace_df if present; otherwise load and parse
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
# Parse timestamps as UTC per instructions
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 1) Compute global P05 and P95 per (cmdb_id, trace_name) using the full series
trace_global = trace_df.groupby(['cmdb_id','trace_name'])['value'].agg(
    count_of_points='count',
    global_P05=lambda x: float(x.quantile(0.05)),
    global_P95=lambda x: float(x.quantile(0.95))
).reset_index()

# 2) Filter trace.csv to the incident window
start = pd.to_datetime('2024-01-25 03:40:18+00:00')
end   = pd.to_datetime('2024-01-25 04:10:18+00:00')
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 3) Join global thresholds to the window rows
trace_window = trace_window.merge(trace_global, on=['cmdb_id','trace_name'], how='left')

# 4) Identify anomaly rows in the window (value >= global_P95 OR value <= global_P05)
trace_window['is_anomaly'] = (trace_window['value'] >= trace_window['global_P95']) | (trace_window['value'] <= trace_window['global_P05'])
anom_window = trace_window[trace_window['is_anomaly']].copy()

# 5) Filter anomalies to trace_name containing 'from_ts-' or 'to_ts-' and referenced service in candidates
# Extract referenced service after 'from_ts-' or 'to_ts-' up to next '.' or end
anom_window['referenced_service'] = anom_window['trace_name'].str.extract(r'(?:from_ts-|to_ts-)(ts-[^.]+)', expand=False)
# Keep only rows where referenced_service is one of the candidate services and non-null
anom_inter = anom_window[anom_window['referenced_service'].isin(candidates)].copy()

# 6) Build distinct inter-candidate trace calls summary
# caller_cmdb_id = cmdb_id (the calling service)
grouped = anom_inter.groupby(['cmdb_id','trace_name','referenced_service'])

summary = grouped.agg(
    count_of_anomalies_in_window=('value','count'),
    earliest_anomaly_timestamp_in_window=('timestamp','min'),
    max_value_in_window=('value','max')
).reset_index().rename(columns={'cmdb_id':'caller_cmdb_id'})

# Attach global_P95 for the caller_cmdb_id + trace_name from trace_global
summary = summary.merge(trace_global[['cmdb_id','trace_name','global_P95']].rename(columns={'cmdb_id':'caller_cmdb_id'}),
                        on=['caller_cmdb_id','trace_name'], how='left')

# Compute severity_ratio = max_value_in_window / global_P95 (handle zero/NaN)
summary['severity_ratio'] = summary.apply(
    lambda r: (r['max_value_in_window'] / r['global_P95']) if pd.notna(r['global_P95']) and r['global_P95'] != 0 else pd.NA,
    axis=1
)

# Sort and limit to top 20 rows by count desc then earliest anomaly asc
summary = summary.sort_values(['count_of_anomalies_in_window','earliest_anomaly_timestamp_in_window'],
                              ascending=[False, True]).head(20)

# Select and order columns as requested
inter_candidate_summary = summary[['caller_cmdb_id','trace_name','referenced_service',
                                   'count_of_anomalies_in_window','earliest_anomaly_timestamp_in_window',
                                   'max_value_in_window','global_P95','severity_ratio']]

# 7) Prepare up to 20 raw anomaly rows for these inter-candidate traces
raw_anoms = anom_inter[['cmdb_id','trace_name','timestamp','value','global_P05','global_P95']].sort_values('timestamp').head(20)

# Final compact outputs
inter_candidate_summary, raw_anoms

```
Out[3]:
```
Result: no inter-candidate anomalous traces found.

- What I did: parsed trace timestamps (UTC), computed global P05 and P95 per (cmdb_id, trace_name) over the full series, filtered to the incident window 2024-01-25 03:40:18 → 2024-01-25 04:10:18 UTC, flagged window points where value >= global_P95 or <= global_P05, and then selected anomaly rows whose trace_name contains "from_ts-" or "to_ts-" and whose referenced service (the substring after that prefix) is one of the five candidates.

- Outcome: both requested outputs are empty:
  1) Distinct inter-candidate trace calls summary: 0 rows.
  2) Raw anomaly rows for inter-candidate traces: 0 rows.

- Interpretation: within the incident window, there were no anomalous trace records that both matched the "from_ts-" / "to_ts-" pattern and referenced one of the candidate services. If you want, I can (a) list all anomalous trace rows in the window regardless of pattern, or (b) show trace_name samples to confirm naming patterns so we can adjust the parsing rule.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [caller_cmdb_id, trace_name, referenced_service, count_of_anomalies_in_window, earliest_anomaly_timestamp_in_window, max_value_in_window, global_P95, severity_ratio]
Index: [], Empty DataFrame
Columns: [cmdb_id, trace_name, timestamp, value, global_P05, global_P95]
Index: [])```
```

In [ ]:
# In[4]:

import pandas as pd
# Reuse existing metric_df if present; otherwise load and parse
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
# Parse timestamps as UTC per instructions
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidate components
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 1) Compute global P05 and P95 per (cmdb_id,kpi_name) using the full series (no window filtering)
metric_candidates = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()
metric_global = metric_candidates.groupby(['cmdb_id','kpi_name'])['value'].agg(
    count_of_points='count',
    global_P05=lambda x: float(x.quantile(0.05)),
    global_P95=lambda x: float(x.quantile(0.95))
).reset_index()

# 2) Filter metric.csv to the incident window
start = pd.to_datetime('2024-01-25 03:40:18+00:00')
end   = pd.to_datetime('2024-01-25 04:10:18+00:00')
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end) & (metric_df['cmdb_id'].isin(candidates))].copy()

# 3) Merge global thresholds onto window rows
metric_window = metric_window.merge(metric_global, on=['cmdb_id','kpi_name'], how='left')

# 4) Mark anomalies: value >= global_P95 OR value <= global_P05
metric_window['is_anomaly'] = False
# Only mark where thresholds are present (non-null)
mask_thresh = metric_window['global_P05'].notna() & metric_window['global_P95'].notna()
metric_window.loc[mask_thresh, 'is_anomaly'] = (
    (metric_window.loc[mask_thresh, 'value'] >= metric_window.loc[mask_thresh, 'global_P95']) |
    (metric_window.loc[mask_thresh, 'value'] <= metric_window.loc[mask_thresh, 'global_P05'])
)

# 5) For each (cmdb_id,kpi_name) in window, identify consecutive anomalous runs (gap <= 60s means consecutive)
anom_rows = metric_window[metric_window['is_anomaly']].copy()
# Ensure sorted for run detection
anom_rows = anom_rows.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

runs_list = []
if not anom_rows.empty:
    # Group and detect runs
    for (cmdb_id, kpi_name), grp in anom_rows.groupby(['cmdb_id','kpi_name'], sort=False):
        grp = grp.sort_values('timestamp').copy()
        # compute gap in seconds between consecutive anomaly points
        grp['gap_sec'] = grp['timestamp'].diff().dt.total_seconds().fillna(0)
        # start a new run if gap > 60
        grp['new_run'] = (grp['gap_sec'] > 60).astype(int)
        grp['run_id'] = grp['new_run'].cumsum()
        # aggregate per run
        run_aggs = grp.groupby('run_id').agg(
            run_start_timestamp=('timestamp','min'),
            run_end_timestamp=('timestamp','max'),
            run_length_points=('timestamp','count'),
            max_value_in_run=('value','max'),
            min_value_in_run=('value','min')
        ).reset_index(drop=True)
        # attach group-level info and global thresholds (these are same for all rows in grp)
        global_P95 = grp['global_P95'].iloc[0]
        global_P05 = grp['global_P05'].iloc[0]
        for _, r in run_aggs.iterrows():
            severity = (r['max_value_in_run'] / global_P95) if (pd.notna(global_P95) and global_P95>0) else pd.NA
            runs_list.append({
                'cmdb_id': cmdb_id,
                'kpi_name': kpi_name,
                'run_start_timestamp': r['run_start_timestamp'],
                'run_end_timestamp': r['run_end_timestamp'],
                'run_length_points': int(r['run_length_points']),
                'run_duration_seconds': int((r['run_end_timestamp'] - r['run_start_timestamp']).total_seconds()),
                'max_value_in_run': r['max_value_in_run'],
                'min_value_in_run': r['min_value_in_run'],
                'global_P95': global_P95,
                'global_P05': global_P05,
                'severity_ratio': severity
            })

# Build runs DataFrame
runs_df = pd.DataFrame(runs_list, columns=[
    'cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length_points',
    'run_duration_seconds','max_value_in_run','min_value_in_run','global_P95','global_P05','severity_ratio'
])

# 6A) Top 20 runs sorted by severity_ratio desc then run_length_points desc
if not runs_df.empty:
    runs_df_sorted = runs_df.sort_values(['severity_ratio','run_length_points'], ascending=[False, False]).reset_index(drop=True)
    top_runs = runs_df_sorted.head(20)
else:
    top_runs = runs_df.head(0)  # empty with same columns

# 6B) Summary table of total_number_of_runs and total_anomalous_points per cmdb_id (one row per candidate)
if not runs_df.empty:
    runs_count = runs_df.groupby('cmdb_id').size().rename('number_of_runs').reset_index()
    anomalous_points = anom_rows.groupby('cmdb_id').size().rename('total_anomalous_points').reset_index()
    summary = pd.merge(pd.DataFrame({'cmdb_id': candidates}), runs_count, on='cmdb_id', how='left')
    summary = pd.merge(summary, anomalous_points, on='cmdb_id', how='left')
    summary['number_of_runs'] = summary['number_of_runs'].fillna(0).astype(int)
    summary['total_anomalous_points'] = summary['total_anomalous_points'].fillna(0).astype(int)
else:
    # no runs: produce zeroed summary for each candidate
    summary = pd.DataFrame({
        'cmdb_id': candidates,
        'number_of_runs': [0]*len(candidates),
        'total_anomalous_points': [0]*len(candidates)
    })

# Compact outputs
top_runs, summary

```
Out[4]:
```
Summary of metric-based anomaly run detection (plain English):

What I did
- Computed global P05 and P95 per (cmdb_id, kpi_name) using each KPI's full series.
- Filtered metric points to the incident window 2024-01-25 03:40:18 → 2024-01-25 04:10:18 UTC.
- Marked window points anomalous when value >= global_P95 or <= global_P05.
- Collapsed consecutive anomalous points into runs (adjacent timestamps with gaps ≤ 60s).
- Reported run-level stats and per-component totals.

Top 20 anomalous runs (highest severity_ratio first) — key highlights
- #1 (most severe): ts-order-service / latency-90 — single-point run at 2024-01-25 03:52:00 UTC. Max=0.362587, global_P95=0.026163 → severity_ratio ≈ 13.86 (large spike vs P95).
- #2: ts-travel-service / diskio — 3-point run from 03:43:00 → 03:45:00 UTC (duration 120s). Max≈156187.39, global_P95≈92526.66 → severity_ratio ≈ 1.69.
- #3: ts-travel-service / cpu — single-point at 03:55:00 UTC, severity_ratio ≈ 1.39.
- #4: ts-train-service / cpu — 2-point run 03:55 → 03:56 (60s), severity_ratio ≈ 1.28.
- Other notable runs include diskio bursts for ts-order-service and ts-auth-service, various latency spikes, and several short single-point runs across cpu, socket, workload, and latency KPIs. Many runs are single-point; a few multi-point runs (length 2–3) exist primarily for diskio and some disk/cpu metrics.

Per-component anomaly summary (candidate components)
- ts-auth-service: 25 runs, 29 anomalous points total.
- ts-order-service: 24 runs, 53 anomalous points total.
- ts-route-service: 25 runs, 53 anomalous points total.
- ts-train-service: 21 runs, 42 anomalous points total.
- ts-travel-service: 22 runs, 53 anomalous points total.

Interpretation / action items
- The single highest-severity event is a large latency-90 spike on ts-order-service (03:52). Investigate request/queue/back-end behavior for that time.
- Disk I/O spikes on ts-travel-service (03:43–03:45) and ts-order-service (03:50–03:52) are multi-point and warrant checking storage or heavy background jobs in that interval.
- Many anomalies are short/single-point; validate whether they reflect transient load bursts or measurement noise.
- Prioritize investigation of ts-order-service, ts-travel-service, ts-train-service and the diskio/latency KPIs showing multi-point and high-severity runs.

If you want, I can (a) expand the top-runs table with exact numeric columns, (b) show the raw anomalous points underlying a specific run, or (c filter results to only multi-point runs.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name       run_start_timestamp         run_end_timestamp  run_length_points  run_duration_seconds  max_value_in_run  min_value_in_run    global_P95    global_P05 severity_ratio
0    ts-order-service  latency-90 2024-01-25 03:52:00+00:00 2024-01-25 03:52:00+00:00                  1                     0          0.362587          0.362587      0.026163      0.021952      13.858783
1   ts-travel-service      diskio 2024-01-25 03:43:00+00:00 2024-01-25 03:45:00+00:00                  3                   120     156187.386623      45393.989544  92526.662965  45437.213070       1.688026
2   ts-travel-service         cpu 2024-01-25 03:55:00+00:00 2024-01-25 03:55:00+00:00                  1                     0          6.517246          6.517246      4.704230      2.832334       1.385401
3    ts-train-service         cpu 2024-01-25 03:55:00+00:00 2024-01-25 03:56:00+00:00                  2                    60          6.227942          5.074753      4.853534      2.614697       1.283177
4    ts-order-service  latency-50 2024-01-25 03:57:00+00:00 2024-01-25 03:57:00+00:00                  1                     0          0.011405          0.011405      0.009991      0.007318       1.141483
5    ts-order-service      diskio 2024-01-25 03:50:00+00:00 2024-01-25 03:52:00+00:00                  3                   120      67461.793264      36101.414748  62325.811292  39965.939120       1.082405
6     ts-auth-service      diskio 2024-01-25 04:06:00+00:00 2024-01-25 04:07:00+00:00                  2                    60      64955.776398      36189.748588  60580.710645  42003.457423       1.072219
7    ts-train-service    workload 2024-01-25 03:49:00+00:00 2024-01-25 03:49:00+00:00                  1                     0         25.015550         25.015550     23.456383     20.138890       1.066471
8     ts-auth-service  latency-50 2024-01-25 03:55:00+00:00 2024-01-25 03:55:00+00:00                  1                     0          0.286236          0.286236      0.270266      0.210002       1.059089
9    ts-order-service         cpu 2024-01-25 04:00:00+00:00 2024-01-25 04:00:00+00:00                  1                     0          3.196939          3.196939      3.025231      1.754465       1.056759
10   ts-route-service    workload 2024-01-25 03:49:00+00:00 2024-01-25 03:49:00+00:00                  1                     0         39.538717         39.538717     37.588085     32.897490       1.051895
11  ts-travel-service  latency-50 2024-01-25 03:57:00+00:00 2024-01-25 03:57:00+00:00                  1                     0          0.022250          0.022250      0.021215      0.017542       1.048777
12    ts-auth-service         cpu 2024-01-25 03:50:00+00:00 2024-01-25 03:50:00+00:00                  1                     0         17.079264         17.079264     16.447620     13.159598       1.038403
13   ts-order-service      socket 2024-01-25 04:01:00+00:00 2024-01-25 04:01:00+00:00                  1                     0         13.166667         13.166667     12.706667     10.403333       1.036201
14  ts-travel-service  latency-90 2024-01-25 04:05:00+00:00 2024-01-25 04:05:00+00:00                  1                     0          0.831241          0.831241      0.805066      0.121813       1.032513
15   ts-route-service  latency-50 2024-01-25 03:45:00+00:00 2024-01-25 03:45:00+00:00                  1                     0          0.004465          0.004465      0.004339      0.003785       1.029042
16   ts-train-service  latency-90 2024-01-25 03:55:00+00:00 2024-01-25 03:56:00+00:00                  2                    60          0.009848          0.009652      0.009577      0.008531       1.028333
17   ts-order-service    workload 2024-01-25 03:59:00+00:00 2024-01-25 03:59:00+00:00                  1                     0          7.951133          7.951133      7.734433      5.191680       1.028018
18   ts-train-service      diskio 2024-01-25 03:43:00+00:00 2024-01-25 03:45:00+00:00                  3                   120      61490.335780      36582.961228  60265.424189  41495.876432       1.020325
19   ts-train-service  latency-50 2024-01-25 03:55:00+00:00 2024-01-25 03:56:00+00:00                  2                    60          0.004573          0.004536      0.004486      0.003876       1.019408,              cmdb_id  number_of_runs  total_anomalous_points
0    ts-auth-service              25                      29
1   ts-order-service              24                      53
2   ts-route-service              25                      53
3   ts-train-service              21                      42
4  ts-travel-service              22                      53)```
```

In [ ]:
# In[5]:

import pandas as pd
import re

# Reuse trace_df if present; otherwise load and parse timestamps
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.to_datetime('2024-01-25 03:40:18+00:00')
end   = pd.to_datetime('2024-01-25 04:10:18+00:00')

# Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Filter to window
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# Keep trace_name containing 'from_ts-' or 'to_ts-' and extract referenced service
pattern = r'(?:from_ts-|to_ts-)(ts-[^\.]+)'
trace_window['referenced_service'] = trace_window['trace_name'].str.extract(pattern, expand=False)

# Filter to rows where referenced_service is one of the candidates (and non-null)
inter = trace_window[trace_window['referenced_service'].isin(candidates)].copy()

# Rename caller column for clarity
inter = inter.rename(columns={'cmdb_id':'caller_cmdb_id'})

# 1) Aggregated compact table grouped by caller_cmdb_id and referenced_service
if not inter.empty:
    agg = inter.groupby(['caller_cmdb_id','referenced_service'], sort=False).agg(
        count_in_window=('value','count'),
        earliest_timestamp_in_window=('timestamp','min'),
        latest_timestamp_in_window=('timestamp','max'),
        median_value_in_window=('value','median'),
        mean_value_in_window=('value','mean')
    ).reset_index()
    agg = agg.sort_values(['count_in_window','earliest_timestamp_in_window'], ascending=[False, True]).head(20)
else:
    agg = pd.DataFrame(columns=['caller_cmdb_id','referenced_service','count_in_window',
                                'earliest_timestamp_in_window','latest_timestamp_in_window',
                                'median_value_in_window','mean_value_in_window'])

# 2) Up to 20 sample raw trace rows for these inter-candidate traces sorted by timestamp asc
if not inter.empty:
    raw_samples = inter[['caller_cmdb_id','trace_name','referenced_service','timestamp','value']].sort_values('timestamp').head(20)
else:
    raw_samples = pd.DataFrame(columns=['caller_cmdb_id','trace_name','referenced_service','timestamp','value'])

# Compact outputs
agg, raw_samples

```
Out[5]:
```
Result: no inter-candidate traces found in the incident window.

What I did
- Parsed trace.csv timestamps as UTC and filtered rows to the incident window 2024-01-25 03:40:18 → 2024-01-25 04:10:18 UTC.
- Selected trace rows whose trace_name contains "from_ts-" or "to_ts-" and parsed the referenced service name after that pattern.
- Kept only rows where the referenced service is one of the five candidates: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service.

Outcome
- Aggregated table by caller_cmdb_id and referenced_service: empty (0 rows).
- Sample raw trace rows for these inter-candidate traces: empty (0 rows).

Interpretation and next steps
- Within the incident window there were no trace rows that both matched the from_ts-/to_ts- pattern and referenced any of the five candidate services.
- If you want, I can (a) list distinct trace_name patterns in the window to confirm naming, (b) show all anomalies regardless of referenced service, or (c) relax the matching rule (e.g., match substrings or different naming formats) to find related calls. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [caller_cmdb_id, referenced_service, count_in_window, earliest_timestamp_in_window, latest_timestamp_in_window, median_value_in_window, mean_value_in_window]
Index: [], Empty DataFrame
Columns: [caller_cmdb_id, trace_name, referenced_service, timestamp, value]
Index: [])```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if they exist; otherwise load and parse timestamps
try:
    metric_df
    trace_df
    log_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    trace_df = pd.read_csv("trace.csv")
    log_df = pd.read_csv("log.csv")

# Parse timestamps as UTC per instructions (idempotent)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp']  = pd.to_datetime(trace_df['timestamp'],  unit='s', utc=True)
log_df['timestamp']    = pd.to_datetime(log_df['timestamp'],    unit='s', utc=True)

# Focus component and incident window
component = "ts-order-service"
start = pd.to_datetime('2024-01-25 03:40:18+00:00')
end   = pd.to_datetime('2024-01-25 04:10:18+00:00')

# ---------------------
# 1) Metrics: global thresholds (full series) for ts-order-service
# ---------------------
metric_full_comp = metric_df[metric_df['cmdb_id'] == component].copy()
metric_global = metric_full_comp.groupby(['cmdb_id','kpi_name'])['value'].agg(
    count_of_points='count',
    global_P05=lambda x: float(x.quantile(0.05)),
    global_P95=lambda x: float(x.quantile(0.95))
).reset_index()

# Windowed metric rows for the component
metric_window = metric_df[(metric_df['cmdb_id'] == component) & (metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()
# merge thresholds
metric_window = metric_window.merge(metric_global, on=['cmdb_id','kpi_name'], how='left')

# mark anomalies (use thresholds computed from full series)
mask_thresh = metric_window['global_P05'].notna() & metric_window['global_P95'].notna()
metric_window['is_anomaly'] = False
metric_window.loc[mask_thresh, 'is_anomaly'] = (
    (metric_window.loc[mask_thresh, 'value'] >= metric_window.loc[mask_thresh, 'global_P95']) |
    (metric_window.loc[mask_thresh, 'value'] <= metric_window.loc[mask_thresh, 'global_P05'])
)

# extract anomalous rows and detect consecutive runs per (kpi_name)
anom_metric_rows = metric_window[metric_window['is_anomaly']].sort_values(['kpi_name','timestamp']).reset_index(drop=True)

runs_list = []
if not anom_metric_rows.empty:
    for kpi, grp in anom_metric_rows.groupby('kpi_name', sort=False):
        g = grp.sort_values('timestamp').copy()
        # compute gap seconds between consecutive anomaly points
        g['gap_sec'] = g['timestamp'].diff().dt.total_seconds().fillna(0)
        # start new run when gap > 60 seconds
        g['new_run'] = (g['gap_sec'] > 60).astype(int)
        g['run_id'] = g['new_run'].cumsum()
        # aggregate per run
        run_aggs = g.groupby('run_id').agg(
            run_start_timestamp=('timestamp','min'),
            run_end_timestamp=('timestamp','max'),
            run_length_points=('timestamp','count'),
            max_value_in_run=('value','max'),
            min_value_in_run=('value','min')
        ).reset_index(drop=True)
        # thresholds (same for this kpi)
        row = g.iloc[0]
        gp95 = row['global_P95']
        gp05 = row['global_P05']
        for _, r in run_aggs.iterrows():
            severity = (r['max_value_in_run'] / gp95) if (pd.notna(gp95) and gp95>0) else np.nan
            runs_list.append({
                'cmdb_id': component,
                'kpi_name': kpi,
                'run_start_timestamp': r['run_start_timestamp'],
                'run_end_timestamp': r['run_end_timestamp'],
                'run_length_points': int(r['run_length_points']),
                'run_duration_seconds': int((r['run_end_timestamp'] - r['run_start_timestamp']).total_seconds()),
                'max_value_in_run': r['max_value_in_run'],
                'min_value_in_run': r['min_value_in_run'],
                'global_P95': gp95,
                'global_P05': gp05,
                'severity_ratio': severity
            })

runs_df = pd.DataFrame(runs_list, columns=[
    'cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length_points',
    'run_duration_seconds','max_value_in_run','min_value_in_run','global_P95','global_P05','severity_ratio'
])

# Sort runs by severity_ratio desc then run_start asc and limit to 20
if not runs_df.empty:
    metric_runs = runs_df.sort_values(['severity_ratio','run_start_timestamp'], ascending=[False, True]).head(20).reset_index(drop=True)
else:
    metric_runs = pd.DataFrame({'note': ['metric runs empty for ts-order-service in window']})

# ---------------------
# 2) Traces: global thresholds (full series) for ts-order-service, then window summary
# ---------------------
trace_full_comp = trace_df[trace_df['cmdb_id'] == component].copy()
trace_global = trace_full_comp.groupby(['cmdb_id','trace_name'])['value'].agg(
    count_of_points='count',
    global_P95=lambda x: float(x.quantile(0.95))
).reset_index()

trace_window = trace_df[(trace_df['cmdb_id'] == component) & (trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

if trace_window.empty:
    trace_summary_ts_order = pd.DataFrame({'note': ['trace window empty for ts-order-service']})
else:
    # compute window stats per trace_name
    trace_stats = trace_window.groupby('trace_name', sort=False).agg(
        count_in_window=('value','count'),
        earliest_timestamp_in_window=('timestamp','min'),
        latest_timestamp_in_window=('timestamp','max'),
        max_value_in_window=('value','max')
    ).reset_index()
    # join global P95
    trace_stats = trace_stats.merge(trace_global[['trace_name','global_P95']], on='trace_name', how='left')
    # compute severity ratio
    def sev_ratio(row):
        gp95 = row['global_P95']
        mv = row['max_value_in_window']
        return (mv / gp95) if (pd.notna(gp95) and gp95>0) else np.nan
    trace_stats['severity_ratio'] = trace_stats.apply(sev_ratio, axis=1)
    # sort by severity_ratio desc and limit 20
    trace_summary_ts_order = trace_stats.sort_values(['severity_ratio','count_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# ---------------------
# 3) Logs: global thresholds for ts-order-service, then window anomaly summary
# ---------------------
log_full_comp = log_df[log_df['cmdb_id'] == component].copy()
log_global = log_full_comp.groupby(['cmdb_id','log_name'])['value'].agg(
    count_of_points_full='count',
    global_P05=lambda x: float(x.quantile(0.05)),
    global_P95=lambda x: float(x.quantile(0.95))
).reset_index()

log_window = log_df[(log_df['cmdb_id'] == component) & (log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)].copy()
# merge thresholds
log_window = log_window.merge(log_global, on=['cmdb_id','log_name'], how='right')  # right to keep all log_names from full series
# After merge, rows from log_global with no window rows will have NaNs in timestamp/value columns

# For anomaly detection we need to consider rows that actually exist in the window; build per-log_name aggregations
# First, derive anomalies among existing window rows (non-null timestamp)
if log_window['timestamp'].notna().any():
    # mark anomalies only where thresholds available and a timestamp exists
    mask_window_rows = log_window['timestamp'].notna() & log_window['global_P05'].notna() & log_window['global_P95'].notna()
    log_window['is_anomaly'] = False
    log_window.loc[mask_window_rows, 'is_anomaly'] = (
        (log_window.loc[mask_window_rows, 'value'] >= log_window.loc[mask_window_rows, 'global_P95']) |
        (log_window.loc[mask_window_rows, 'value'] <= log_window.loc[mask_window_rows, 'global_P05'])
    )
else:
    log_window['is_anomaly'] = False

# Aggregate per log_name: use global counts and thresholds, compute window-based stats
# group by log_name (from log_global)
log_names = log_global['log_name'].unique()
rows = []
for ln in log_names:
    g_global = log_global[log_global['log_name']==ln].iloc[0]
    gp05 = g_global['global_P05']
    gp95 = g_global['global_P95']
    count_full = int(g_global['count_of_points_full'])
    # window rows for this log_name
    gw = log_window[log_window['log_name']==ln]
    # consider only rows that actually have timestamp (window present)
    gw_rows = gw[gw['timestamp'].notna()]
    count_anom = int(gw_rows['is_anomaly'].sum()) if not gw_rows.empty else 0
    earliest_anom_ts = gw_rows[gw_rows['is_anomaly']]['timestamp'].min() if (not gw_rows.empty and gw_rows['is_anomaly'].any()) else pd.NaT
    max_value_in_window = gw_rows['value'].max() if not gw_rows.empty else np.nan
    min_value_in_window = gw_rows['value'].min() if not gw_rows.empty else np.nan
    rows.append({
        'log_name': ln,
        'count_of_points_full': count_full,
        'global_P05': gp05,
        'global_P95': gp95,
        'count_anomalies_in_window': count_anom,
        'earliest_anomaly_timestamp_in_window': earliest_anom_ts,
        'max_value_in_window': max_value_in_window,
        'min_value_in_window': min_value_in_window
    })

log_summary_ts_order = pd.DataFrame(rows, columns=[
    'log_name','count_of_points_full','global_P05','global_P95',
    'count_anomalies_in_window','earliest_anomaly_timestamp_in_window','max_value_in_window','min_value_in_window'
])

# Compact results: ensure bounded to 20 rows
if isinstance(metric_runs, pd.DataFrame) and 'note' in metric_runs.columns and len(metric_runs)==1:
    metric_runs_out = metric_runs  # note DataFrame indicating emptiness
else:
    metric_runs_out = metric_runs.head(20)

trace_summary_out = trace_summary_ts_order.head(20) if isinstance(trace_summary_ts_order, pd.DataFrame) else trace_summary_ts_order
log_summary_out = log_summary_ts_order.head(20) if not log_summary_ts_order.empty else pd.DataFrame({'note':['log summary empty for ts-order-service']})

# Return the three compact outputs (variables displayed)
metric_runs_out, trace_summary_out, log_summary_out

```
Out[6]:
```
Summary for ts-order-service (plain English):

Metric fault runs (top results)
- Highest-severity run: latency-90 spike at 2024-01-25 03:52:00 UTC (single-point run). Max=0.3626, global_P95=0.02616 → severity_ratio ≈ 13.86. Investigate this large p90 latency spike.
- Significant multi-point disk I/O run: diskio from 03:50:00 → 03:52:00 UTC (3 points, 120s). Max≈67461.79, global_P95≈62325.81 → severity_ratio ≈ 1.08. Check storage or heavy I/O jobs in 03:50–03:52.
- Other notable runs (all within the incident window):
  - latency-50 single spike at 03:57 (severity ≈ 1.14).
  - cpu single/multi-point runs at 04:00 (single) and 03:49–03:50 (2-point, severity ≈ 1.006).
  - socket, workload, mem runs with mild severity ratios just above 1.0 (several single-point runs around 03:55–04:06).
- Several runs are single-point; a few multi-point runs (length 2–3) exist mainly for diskio and cpu/disk metrics.

Trace summary (windowed, with global P95 comparison; top by severity_ratio)
- Highest severity traces (examples from top 20):
  - trace.from_ts-preserve-service.duration_mean — max 0.04946, global_P95 0.02465 → severity ≈ 2.01.
  - trace.from_ts-seat-service.duration_p95 — max 0.04049, global_P95 0.02301 → severity ≈ 1.76.
  - trace.from_ts-preserve-service.duration_p95 — severity ≈ 1.29.
  - trace.from_ts-preserve-service.row_count — max 10.0, global_P95 7.8 → severity ≈ 1.28.
  - trace.from_root.row_count and to_ts-station-service.row_count — max 18.0, global_P95 14.4 → severity ≈ 1.25.
  - trace.from_ts-order-service.row_count and to_ts-order-service.row_count — large counts (max 649.0) vs global_P95 590.6 → severity ≈ 1.10.
- In short: several outgoing/incoming trace durations and row_count metrics exceed their global P95 in the window (notably calls involving preserve-service and seat-service and large row_count for order-service itself). Investigate inter-service call latency/throughput around these traces.

Log summary (windowed, using global P05/P95 from full series)
- log.error_count: count_of_points_full = 25, global_P05 = global_P95 = 0.0. All 25 window points are flagged (value = 0). This indicates the error_count series is uniformly zero for the full series—interpret carefully (could be genuine or a metric artifact).
- log.row_count: count_of_points_full = 25, global_P05 = 302.0, global_P95 = 465.2. In the window there are 4 anomalous points (value >= P95 or <= P05); earliest anomaly at 2024-01-25 03:44:00 UTC. Observed max in window = 467, min = 116. Check traffic/ingest spikes and drops around ~03:44 and subsequent times.

Overall concise recommendations
- Immediately investigate ts-order-service latency-90 spike at 03:52 (very high severity).
- Investigate disk I/O activity around 03:50–03:52 (multi-point diskio run).
- Review inter-service traces flagged above (preserve-service, seat-service, and high row_count on order-service) for upstream/downstream impact.
- Treat the all-zero log.error_count carefully — confirm whether that metric is valid or misreported.
- If you want, I can show the raw points underlying any specific run or trace for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id    kpi_name       run_start_timestamp         run_end_timestamp  run_length_points  run_duration_seconds  max_value_in_run  min_value_in_run    global_P95    global_P05  severity_ratio
0   ts-order-service  latency-90 2024-01-25 03:52:00+00:00 2024-01-25 03:52:00+00:00                  1                     0      3.625866e-01      3.625866e-01  2.616295e-02  2.195181e-02       13.858783
1   ts-order-service  latency-50 2024-01-25 03:57:00+00:00 2024-01-25 03:57:00+00:00                  1                     0      1.140464e-02      1.140464e-02  9.991070e-03  7.318014e-03        1.141483
2   ts-order-service      diskio 2024-01-25 03:50:00+00:00 2024-01-25 03:52:00+00:00                  3                   120      6.746179e+04      3.610141e+04  6.232581e+04  3.996594e+04        1.082405
3   ts-order-service         cpu 2024-01-25 04:00:00+00:00 2024-01-25 04:00:00+00:00                  1                     0      3.196939e+00      3.196939e+00  3.025231e+00  1.754465e+00        1.056759
4   ts-order-service      socket 2024-01-25 04:01:00+00:00 2024-01-25 04:01:00+00:00                  1                     0      1.316667e+01      1.316667e+01  1.270667e+01  1.040333e+01        1.036201
5   ts-order-service    workload 2024-01-25 03:59:00+00:00 2024-01-25 03:59:00+00:00                  1                     0      7.951133e+00      7.951133e+00  7.734433e+00  5.191680e+00        1.028018
6   ts-order-service      diskio 2024-01-25 03:57:00+00:00 2024-01-25 03:57:00+00:00                  1                     0      6.313239e+04      6.313239e+04  6.232581e+04  3.996594e+04        1.012941
7   ts-order-service  latency-90 2024-01-25 04:04:00+00:00 2024-01-25 04:04:00+00:00                  1                     0      2.642264e-02      2.642264e-02  2.616295e-02  2.195181e-02        1.009926
8   ts-order-service         cpu 2024-01-25 03:49:00+00:00 2024-01-25 03:50:00+00:00                  2                    60      3.044285e+00      1.541157e+00  3.025231e+00  1.754465e+00        1.006298
9   ts-order-service  latency-50 2024-01-25 04:04:00+00:00 2024-01-25 04:04:00+00:00                  1                     0      1.002961e-02      1.002961e-02  9.991070e-03  7.318014e-03        1.003857
10  ts-order-service    workload 2024-01-25 04:03:00+00:00 2024-01-25 04:03:00+00:00                  1                     0      7.757767e+00      7.757767e+00  7.734433e+00  5.191680e+00        1.003017
11  ts-order-service         mem 2024-01-25 04:06:00+00:00 2024-01-25 04:07:00+00:00                  2                    60      2.815767e+08      2.810492e+08  2.809354e+08  2.695580e+08        1.002283
12  ts-order-service      socket 2024-01-25 04:06:00+00:00 2024-01-25 04:06:00+00:00                  1                     0      1.271667e+01      1.271667e+01  1.270667e+01  1.040333e+01        1.000787
13  ts-order-service         mem 2024-01-25 03:43:00+00:00 2024-01-25 03:44:00+00:00                  2                    60      2.694508e+08      2.694236e+08  2.809354e+08  2.695580e+08        0.959120
14  ts-order-service  latency-90 2024-01-25 03:44:00+00:00 2024-01-25 03:44:00+00:00                  1                     0      2.194644e-02      2.194644e-02  2.616295e-02  2.195181e-02        0.838837
15  ts-order-service      socket 2024-01-25 03:55:00+00:00 2024-01-25 03:55:00+00:00                  1                     0      1.038333e+01      1.038333e+01  1.270667e+01  1.040333e+01        0.817156
16  ts-order-service  latency-90 2024-01-25 03:50:00+00:00 2024-01-25 03:50:00+00:00                  1                     0      2.107269e-02      2.107269e-02  2.616295e-02  2.195181e-02        0.805440
17  ts-order-service      socket 2024-01-25 03:46:00+00:00 2024-01-25 03:46:00+00:00                  1                     0      1.023333e+01      1.023333e+01  1.270667e+01  1.040333e+01        0.805352
18  ts-order-service  latency-50 2024-01-25 03:50:00+00:00 2024-01-25 03:50:00+00:00                  1                     0      7.287098e-03      7.287098e-03  9.991070e-03  7.318014e-03        0.729361
19  ts-order-service  latency-50 2024-01-25 03:53:00+00:00 2024-01-25 03:53:00+00:00                  1                     0      7.210625e-03      7.210625e-03  9.991070e-03  7.318014e-03        0.721707,                                            trace_name  count_in_window earliest_timestamp_in_window latest_timestamp_in_window  max_value_in_window  global_P95  severity_ratio
0        trace.from_ts-preserve-service.duration_mean               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00             0.049459    0.024645        2.006805
1             trace.from_ts-seat-service.duration_p95               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00             0.040490    0.023011        1.759597
2         trace.from_ts-preserve-service.duration_p95               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00             0.081478    0.063373        1.285695
3            trace.from_ts-preserve-service.row_count               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00            10.000000    7.800000        1.282051
4                           trace.from_root.row_count               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00            18.000000   14.400000        1.250000
5               trace.to_ts-station-service.row_count               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00            18.000000   14.400000        1.250000
6            trace.from_ts-seat-service.duration_mean               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00             0.011889    0.009887        1.202579
7           trace.from_ts-order-service.duration_mean               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00             0.034498    0.029281        1.178163
8             trace.to_ts-order-service.duration_mean               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00             0.034498    0.029281        1.178163
9         trace.from_ts-security-service.duration_p95               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00             0.023840    0.021254        1.121651
10               trace.from_ts-seat-service.row_count               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00           150.000000  133.800000        1.121076
11     trace.from_ts-inside-payment-service.row_count               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00            13.000000   11.600000        1.120690
12              trace.from_ts-order-service.row_count               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00           649.000000  590.600000        1.098882
13                trace.to_ts-order-service.row_count               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00           649.000000  590.600000        1.098882
14             trace.from_ts-travel-service.row_count               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00            71.000000   66.600000        1.066066
15       trace.from_ts-security-service.duration_mean               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00             0.017277    0.016293        1.060368
16         trace.from_ts-travel-service.duration_mean               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00             0.010523    0.010025        1.049750
17  trace.from_ts-inside-payment-service.duration_p95               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00             0.008975    0.008552        1.049523
18                       trace.from_root.duration_p95               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00             1.081379    1.042775        1.037020
19          trace.from_ts-travel-service.duration_p95               25    2024-01-25 03:43:00+00:00  2024-01-25 04:07:00+00:00             0.021231    0.020504        1.035446,           log_name  count_of_points_full  global_P05  global_P95  count_anomalies_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  min_value_in_window
0  log.error_count                    25         0.0         0.0                         25            2024-01-25 03:43:00+00:00                    0                    0
1    log.row_count                    25       302.0       465.2                          4            2024-01-25 03:44:00+00:00                  467                  116)```
```